# Explore: Order items

`olist_order_items_dataset.csv`

In [1]:
import pandas as pd

items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")  # needed for the gap check later
items.shape

(112650, 7)

112650 rows, 7 columns. More rows than orders (112650 vs 99441), so some orders have multiple items.

In [2]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


Columns cover the order specifics: order, item, product and seller ID, shipping deadline date, order cost and delivery cost. `order_item_id` is the item's position in the order (1 = first item), that is why one `order_id` can appear on several rows.

In [3]:
items.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Order, product and seller IDs and the shipping deadline are `str`. `order_item_id` is `int64` because it is the item number within the order. Price and freight are `float64` because they are currency values.

In [4]:
items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

No missing values.

In [5]:
items.groupby("order_id").size().describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

Most orders have 1 item. The average is slightly above 1, and the largest order has 21 items.

In [6]:
orders_without_items = set(orders["order_id"]) - set(items["order_id"])
len(orders_without_items)

775

775 orders have no item rows at all.

In [7]:
orders[orders["order_id"].isin(orders_without_items)]["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Almost all of the no item orders are unavailable (603) or canceled (164): orders that failed before items existed. A few are early-stage, plus 1 marked shipped which looks wrong.

In [8]:
items["price"].describe()

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

Item prices range from 0.85 to 6735, with a median of 74.99. The average (120.65) sits above the median, so a few expensive items pull it up.